# Lesson 8 : Harness agent in MAF

Microsoft Agent Framework provides fully-featured pre-built agent template for production patterns, called harness agent - in which the following features are assembled by default. :

- Default harness instruction out of the box
- Hosted web search tool out of the box 
- ```InMemoryHistoryProvider``` for the memory history
- ```ContextWindowCompactionStrategy``` in before-strategy and ```ToolResultCompactionStrategy``` in after-strategy
- ```TodoProvider``` for todo list management
- ```AgentModeProvider``` for plan/execute mode tracking

You can also add features or customize existing features if required.<br>
For instance, the following is the optional settings in harness agent.

- Function invocation, when ```tools``` property is provided.
- ```MemoryContextProvider``` for persistent file-based memory, when ```memory_store``` property is provided. (Instantiated by ```MemoryContextProvider(store=memory_store)```.)
- ```SkillsProvider``` for agent skills, when ```skills_provider``` or ```skills_paths``` property is provided.

> Note : For the latest detailed implementation, please refer to the source code of ```create_harness_agent()```.

## Harness Agent (out of the box)

In the first example, we explore harness agent with default settings.

Before staring, same as in Lesson 1, we create a client as follows to run on Microsoft Foundry.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Harness agent is instantiated by ```create_harness_agent()```.<br>
In the following code, we create a harness agent with minimal settings (without any additional settings).

In the default harness instruction, it tells to break the work into clear steps.<br>
If you want to add instructions while maintaining the harness instruction, specify ```agent_instructions``` property as follows.

> Note : The default harness instruction can also be modified (customized) by specifying ```harness_instructions``` property.

In [2]:
from agent_framework import create_harness_agent

agent = create_harness_agent(
    client=client,
    agent_instructions=(
        "You are a helpful weather assistant. "
        "Keep the history of locations queried by users in a memory file called `locations.md`."
    ),
    name="HarnessAgent",
    max_context_window_tokens=128000,
    max_output_tokens=16384,
)

Let's run this harness agent.

The default location for memories in harness agents is ```agent-file-memory``` directory.<br>
Because the instruction is configured to save the location queried by the user to ```locations.md```, you can see ```locations.md``` (in which "Osaka" is recorded) in the session folder on ```agent-file-memory``` directory.

The agent will often return the response which tells the user to clarify instructions, because the default harness instruction indicates to break the work into clear steps. (Please compare with the response in Lesson 4.)

In [3]:
session = agent.create_session()
result = await agent.run(
    "Tell me the weather in Osaka today.",
    session=session,
)
print(result.text)

**Osaka weather for today (Osaka time/JST): Friday, June 5, 2026**

- **Outlook:** Rain early, becoming cloudy later (“RAIN, CLOUDY LATER”). ([data.jma.go.jp](https://www.data.jma.go.jp/multi/yoho/yoho_detail.html?code=270000&lang=en))  
- **Temperature:** **High 26°C / Low 19°C**. ([data.jma.go.jp](https://www.data.jma.go.jp/multi/yoho/yoho_detail.html?code=270000&lang=en))  
- **Chance of precipitation (JMA time blocks):**
  - **00–06:** 50%
  - **06–12:** 50%
  - **12–18:** 20%
  - **18–24:** 20% ([data.jma.go.jp](https://www.data.jma.go.jp/multi/yoho/yoho_detail.html?code=270000&lang=en))  

If you want, tell me what time range you’ll be out (morning/afternoon/evening) and I’ll translate that into a quick “umbrella or not” suggestion.


## Harness agent with custom settings

In the next example, we change harness agent to use only local functiosn instead of using web search tool, as we did in the previous examples.<br>
Furthermore, we configure file access provider to save the result in files.

In the following code, we supress web search tool by specifying ```disable_web_search=True```, and set local functions by specifying ```tools``` property.

To set file access provider, we set ```FileSystemAgentFileStore``` in ```file_access_store``` property as follows.<br>
By default, any operations for file access needs approval by the user. To run unattended (automatically approve), I have also specified ```auto_approval_rules``` property as follows.

> Note : For available properties, see the implementation of ```create_harness_agent()```.

In [4]:
from agent_framework import tool
from typing import Annotated
from pydantic import Field
from random import randint

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

In [5]:
from agent_framework import FileSystemAgentFileStore, FileAccessProvider

agent = create_harness_agent(
    client=client,
    name="HarnessAgent",
    max_context_window_tokens=128000,
    max_output_tokens=16384,
    disable_web_search=True,
    tools=[get_weather, get_temperature],
    file_access_store=FileSystemAgentFileStore("working"),
    auto_approval_rules=[FileAccessProvider.all_tools_auto_approval_rule],
)

Now let's invoke the harness agent, and see the result in ```osaka.txt``` as follows.

In [6]:
session = agent.create_session()
result = await agent.run(
    "Please write the weather and temperature for Osaka to osaka.txt.",
    session=session,
)
print(result.text)

Wrote `osaka.txt` with:

- Weather in Osaka: cloudy  
- Temperature in Osaka: 11 degrees


In [7]:
with open("working/osaka.txt", "r") as file:
    file_content = file.read()
    print(file_content)

Weather in Osaka: cloudy
Temperature in Osaka: 11 degrees

